In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm
import datamol as dm

In [3]:
df = pd.read_csv("../data/Vapor_pressures.csv")
df.head()

,Name,CAS-RN,Formula,SMILES,T,Pvap,KDB_ID,Class,Subclass
0,METHANE,74-82-8,CH4,C,90.670000,11.702099,1,Paraffinic Hydrocarbons,n-Alkanes
1,METHANE,74-82-8,CH4,C,94.115172,17.886999,1,Paraffinic Hydrocarbons,n-Alkanes
2,METHANE,74-82-8,CH4,C,97.560345,26.489535,1,Paraffinic Hydrocarbons,n-Alkanes
3,METHANE,74-82-8,CH4,C,101.005517,38.137051,1,Paraffinic Hydrocarbons,n-Alkanes
4,METHANE,74-82-8,CH4,C,104.450690,53.534190,1,Paraffinic Hydrocarbons,n-Alkanes


In [5]:
def get_fp(mol, fp_type='topological'):
    return dm.to_fp(mol, fp_type=fp_type)

df['Mol'] = df['SMILES'].apply(dm.to_mol)
df['Fingerprint'] = df['Mol'].apply(get_fp)
df.head()

,Name,CAS-RN,Formula,SMILES,T,Pvap,KDB_ID,Class,Subclass,Mol,Fingerprint
0,METHANE,74-82-8,CH4,C,90.670000,11.702099,1,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x000001BAB80...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,METHANE,74-82-8,CH4,C,94.115172,17.886999,1,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x000001BAB80...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,METHANE,74-82-8,CH4,C,97.560345,26.489535,1,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x000001BAB80...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,METHANE,74-82-8,CH4,C,101.005517,38.137051,1,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x000001BAB80...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,METHANE,74-82-8,CH4,C,104.450690,53.534190,1,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x000001BAB80...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


In [8]:
df["ln_P"] = np.log(df.Pvap)


In [9]:
df.columns.tolist()

['Name',
 'CAS-RN',
 'Formula',
 'SMILES',
 'T',
 'Pvap',
 'KDB_ID',
 'Class',
 'Subclass',
 'Mol',
 'Fingerprint',
 'ln_P']

In [13]:
# Splitting the dataset in such a way that the training data/compounds should not be in test data/compounds

from sklearn.model_selection import train_test_split
chemical_classes = df["Class"].unique()
# 3 different dataset for storing the names of the compounds
train_comps, valid_comps, test_comps = [], [], []
for cl in chemical_classes:
    # take all the data belonging to cl class
    df_cl = df[df["Class"]==cl]
    # Get unique compound for this data belonging to cl class
    unique_compound_cl = df_cl["Name"].unique().tolist()
    # Split into train and test
    train_all, comp_test = train_test_split(unique_compound_cl, test_size = 0.20, random_state = 0)
    # Split the train all to train and valid 
    comp_train, comp_valid = train_test_split(train_all, test_size = 0.20, random_state = 1)

    train_comps.extend(list(comp_train))
    valid_comps.extend(list(comp_valid))
    test_comps.extend(list(comp_test))

In [14]:
print(train_comps)

['4-METHYLHEPTANE', 'ISOBUTANE', '3,3-DIMETHYLHEPTANE', '2,2-DIMETHYLOCTANE', '3-METHYLHEXANE', 'N-HEXADECANE', 'N-BUTANE', '3,4-DIMETHYLHEPTANE', '3-METHYLNONANE', '3,3,4-TRIMETHYLHEPTANE', '4-METHYLNONANE', 'N-HEXATRIACONTANE', 'N-DODECANE', '2,2,6-TRIMETHYLHEPTANE', '3-METHYL-3-ETHYLPENTANE', '2,4-DIMETHYL-3-ETHYLHEXANE', '2,4-DIMETHYLHEPTANE', '3-ETHYLHEXANE', '4-ETHYLOCTANE', '2,4,5-TRIMETHYLHEPTANE', '2,4-DIMETHYL-3-ISOPROPYLPENTANE', '2,3,3-TRIMETHYLHEPTANE', '2,3,4,4-TETRAMETHYLHEXANE', '2-METHYL-4-ETHYLHEPTANE', '2,3,3,5-TETRAMETHYLHEXANE', '3,3,5-TRIMETHYLHEPTANE', '2,5-DIMETHYLHEXANE', '2,2,4-TRIMETHYLHEXANE', 'N-PENTADECANE', '2,3,6-TRIMETHYLHEPTANE', 'N-EICOSANE', '2,2,4-TRIMETHYLPENTANE', '2,2,3,3-TETRAMETHYLBUTANE', '2-METHYLHEXANE', 'N-UNDECANE', 'N-TRICOSANE', 'N-TRITRIACONTANE', '3-METHYLHEPTANE', '2,3,4-TRIMETHYLHEPTANE', '2,3,5-TRIMETHYLHEXANE', '2,2,3,4,4-PENTAMETHYLPENTANE', '3,3-DIMETHYL-4-ETHYLHEXANE', '4,5-DIMETHYLOCTANE', 'N-TRIDECANE', '2,3-DIMETHYL-4-ETHYLHE

In [19]:
df_train = df[df["Name"].isin(train_comps)].reset_index()
df_valid = df[df["Name"].isin(valid_comps)].reset_index()
df_test = df[df["Name"].isin(test_comps)].reset_index()

In [20]:
df_train.head()

,index,Name,CAS-RN,Formula,SMILES,T,Pvap,KDB_ID,Class,Subclass,Mol,Fingerprint,ln_P
0,90,N-BUTANE,106-97-8,C4H10,CCCC,134.860000,0.000669,4,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x000001BAB82...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",-7.309849
1,91,N-BUTANE,106-97-8,C4H10,CCCC,144.871035,0.003877,4,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x000001BAB82...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",-5.552663
2,92,N-BUTANE,106-97-8,C4H10,CCCC,154.882069,0.017450,4,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x000001BAB82...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",-4.048429
3,93,N-BUTANE,106-97-8,C4H10,CCCC,164.893103,0.064001,4,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x000001BAB82...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",-2.748860
4,94,N-BUTANE,106-97-8,C4H10,CCCC,174.904138,0.198505,4,Paraffinic Hydrocarbons,n-Alkanes,<rdkit.Chem.rdchem.Mol object at 0x000001BAB82...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",-1.616940


In [21]:
print(df_train.shape)
print(df_valid.shape)
print(df_test.shape)

(26310, 13)
(6810, 13)
(8610, 13)


In [22]:
df_train.columns.tolist()

['index',
 'Name',
 'CAS-RN',
 'Formula',
 'SMILES',
 'T',
 'Pvap',
 'KDB_ID',
 'Class',
 'Subclass',
 'Mol',
 'Fingerprint',
 'ln_P']

In [23]:
# Standardized the temp. values
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(df_train['T'].to_numpy().reshape(-1,1))

df_train['T_scaled'] = scaler.transform(df_train['T'].to_numpy().reshape(-1,1))
df_valid['T_scaled'] = scaler.transform(df_valid['T'].to_numpy().reshape(-1,1))
df_test['T_scaled'] = scaler.transform(df_test['T'].to_numpy().reshape(-1,1))

In [24]:
df["Fingerprint"].tolist()

[array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0, 0, 0], shape=(2048,), dtype=uint8),
 array([0, 0, 0, ..., 0,

In [25]:
def getX_y(A,):
    X = np.vstack(A["Fingerprint"].tolist())
    T = A["T_scaled"].to_numpy().reshape(-1,1)
    X = np.hstack((X, T))
    y = A['ln_P'].to_numpy().reshape(-1,1)
    return X, y

In [26]:
X_valid, y_valid = getX_y(df_valid)
X_test,  y_test  = getX_y(df_test)
X_train, y_train = getX_y(df_train)

In [27]:
X_train.shape, y_train.shape

((26310, 2049), (26310, 1))

In [28]:
X_test.shape, X_valid.shape, y_test.shape, y_valid.shape

((8610, 2049), (6810, 2049), (8610, 1), (6810, 1))

In [36]:
# Let's start building the DecisionTreeRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_absolute_percentage_error as MAPE
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score as R2

model_Tree = DecisionTreeRegressor(random_state=42)
model_Tree.fit(X_train, y_train.ravel())
y_train_Tree = model_Tree.predict(X_train)
print("For training data: ")
print("Train MAE:", MAE(y_train, y_train_Tree))
print("MAPE: ", MAPE(y_train, y_train_Tree))
print("MAE: ", MAE(y_train, y_train_Tree))
print("R2: ", R2(y_train, y_train_Tree))
print("RMSE: ", root_mean_squared_error(y_train, y_train_Tree))
y_valid_Tree = model_Tree.predict(X_valid)
print("For validation data: ")
print("MAPE: ", MAPE(y_valid, y_valid_Tree))
print("MAE: ", MAE(y_valid, y_valid_Tree))
print("R2: ", R2(y_valid, y_valid_Tree))
print("RMSE: ", root_mean_squared_error(y_valid, y_valid_Tree))


For training data: 
Train MAE: 0.001971611908194476
MAPE:  0.001742076828771046
MAE:  0.001971611908194476
R2:  0.9997264430769087
RMSE:  0.07060296178486072
For validation data: 
MAPE:  1.7741155063597316
MAE:  1.7445972673990235
R2:  0.5532660763556234
RMSE:  2.74224839656232


In [38]:
# Make a function for calling the model and fitting it 
import time
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import root_mean_squared_error
from sklearn.metrics import r2_score as R2

def fit_predict(model, X_train, y_train, X_valid):
    start = time.time()
    model.fit(X_train, y_train.ravel())
    fit_time = time.time() - start
    return model, model.predict(X_train), model.predict(X_valid), fit_time

def model_metrics(name, y_train, y_train_pred, y_valid, y_valid_pred, fit_time=None):
    y_train, y_train_pred = np.ravel(y_train), np.ravel(y_train_pred)
    y_valid, y_valid_pred = np.ravel(y_valid), np.ravel(y_valid_pred)
    return {
        "model": name,
        "train_MAE": MAE(y_train, y_train_pred),
        "train_RMSE": root_mean_squared_error(y_train, y_train_pred),
        "train_R2": R2(y_train, y_train_pred),
        "valid_MAE": MAE(y_valid, y_valid_pred),
        "valid_RMSE": root_mean_squared_error(y_valid, y_valid_pred),
        "valid_R2": R2(y_valid, y_valid_pred),
        "fit_time_s": fit_time,
    }

In [39]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

models_to_try = {
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=-1),
    "XGBoost": XGBRegressor(random_state=42, n_jobs=-1),
}

results = []
for name, mdl in models_to_try.items():
    fitted, p_tr, p_va, t = fit_predict(mdl, X_train, y_train, X_valid)
    results.append(model_metrics(name, y_train, p_tr, y_valid, p_va, t))

pd.DataFrame(results)

,model,train_MAE,train_RMSE,train_R2,valid_MAE,valid_RMSE,valid_R2,fit_time_s
0,Decision Tree,0.001972,0.070603,0.999726,1.744597,2.742248,0.553266,40.740460
1,Random Forest,0.314885,0.582203,0.981398,1.437179,2.254949,0.697929,252.344966
2,XGBoost,0.783024,1.339202,0.901578,1.323195,2.110358,0.735426,5.543770
